# Residential land values: 2023, 2026, and change

Three related maps using the SDK continuous choropleth API.

This is an editable worked example of [`parcel_land_value_change.py`](../src/graphics/parcel_land_value_change.py). It runs Python SDK calls directly—no website, CLI subprocess, or registered build dispatcher. The canonical definition remains the publishing source of truth.

**What the numbers mean:** Uses the same matched residential cohort for all three maps. The 2026 values use the existing isolated-spike smoothing rule. Missing parcels are outside the comparison, not zero change.

Start with **Run All**, inspect the data table and preview, then change the title or a visual encoding in step 4. Parcel examples load the full city and can take several minutes.


## 1. Open the libraries

Use the environment in [README.md](README.md). Paths below locate the checkout, not a personal machine.


In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder or anywhere inside the Detroit checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "strongtowns-data.lock.json").is_file()
             and (p / "projects/graphics/src/graphics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter inside the strongtowns-detroit checkout; see README.md.")
sys.path.insert(0, str(ROOT / "src"))
GRAPHICS = ROOT / "projects/graphics"
FORUM = ROOT / "projects/detroit-land-use-forum"
import strongtowns_graphics as graphics_sdk
if not hasattr(graphics_sdk, "GraphicInput"):
    raise RuntimeError(
        "This kernel has an older graphics SDK. Restart Jupyter with the uv command "
        "in README.md so it uses this project's locked dependencies."
    )
from IPython.display import SVG, display
from strongtowns_graphics import (
    GraphicFormat, GRAPHIC_FORMAT_SPECS, render_graphic_canvas,
    render_graphic_svg, write_graphic_bundle,
)

NOTEBOOK_NAME = 'parcel_land_value_change'


In [ ]:
import sys

from functools import lru_cache

from pathlib import Path

import geopandas as gpd

import numpy as np

import polars as pl

import pyogrio

sys.path.insert(0, str(GRAPHICS))

from basemap import load_detroit_basemap  # noqa: E402

from strongtowns_graphics import (  # noqa: E402
    ContinuousChoroplethScale,
    GraphicInput,
    graphic_definition,
    parcel_choropleth_map,
)

from strongtowns_data.pipelines.land_values import (  # noqa: E402
    LandValueSmoothing,
    SmoothingMode,
    apply_land_value_smoothing,
)

NO_DATA = "#e4dfd6"

VALUE_SCALE = ContinuousChoroplethScale(
    minimum=3_000,
    maximum=160_000,
    colors=("#b8d2ef", "#72a7df", "#fff1c9", "#ffbd59", "#ef8545", "#8f1d2d"),
    ticks=(
        (3_000, "≤$3K"),
        (10_000, "$10K"),
        (30_000, "$30K"),
        (100_000, "$100K"),
        (160_000, "≥$160K"),
    ),
    logarithmic=True,
)

CHANGE_SCALE = ContinuousChoroplethScale(
    minimum=-60_000,
    maximum=60_000,
    midpoint=0,
    colors=("#356daf", "#91b9e6", "#f5eddd", "#ffbd59", "#e8783d", "#9d2132"),
    ticks=(
        (-60_000, "≤−$60K"),
        (-30_000, "−$30K"),
        (0, "$0"),
        (30_000, "+$30K"),
        (60_000, "≥+$60K"),
    ),
)

def comparison_frame(
    parcels_path: Path,
    current_source: Path,
    history_source: Path,
) -> gpd.GeoDataFrame:
    parcels = gpd.read_file(parcels_path, columns=["parcel_id", "geometry"])
    parcels = parcels.drop_duplicates("parcel_id").to_crs("EPSG:3857")
    centroids = parcels.geometry.centroid
    coordinates = pl.DataFrame({
        "parcel_id": parcels["parcel_id"].astype(str),
        "x": centroids.x,
        "y": centroids.y,
    })

    current_rows = pyogrio.read_dataframe(
        current_source,
        read_geometry=False,
        columns=["parcel_id", "amt_land_value", "total_square_footage"],
    )
    current = pl.from_pandas(current_rows).select(
        pl.col("parcel_id").cast(pl.String),
        pl.col("amt_land_value").cast(pl.Float64, strict=False).alias("land_value"),
        pl.col("total_square_footage")
        .cast(pl.Float64, strict=False)
        .alias("parcel_area_sqft"),
    ).join(coordinates, on="parcel_id", how="inner")
    current = apply_land_value_smoothing(
        current,
        config=LandValueSmoothing(mode=SmoothingMode.ISOLATED_SPIKES),
    ).select("parcel_id", "parcel_area_sqft", "selected_land_value")

    history = pl.read_csv(
        history_source,
        schema_overrides={"parcel_num": pl.String},
        infer_schema_length=10_000,
    ).select(
        pl.col("parcel_num").str.strip_chars().alias("parcel_id"),
        pl.col("land_value").cast(pl.Float64, strict=False).alias("land_value_2023"),
    ).filter(pl.col("land_value_2023") > 0)
    conflicting = history.group_by("parcel_id").agg(
        pl.col("land_value_2023").n_unique().alias("values")
    ).filter(pl.col("values") > 1)
    if conflicting.height:
        raise ValueError("2023 source has conflicting land values for a parcel")
    history = history.unique("parcel_id", keep="first")

    comparison = history.join(current, on="parcel_id", how="inner").filter(
        pl.col("parcel_area_sqft").is_finite()
        & (pl.col("parcel_area_sqft") > 0)
        & pl.col("selected_land_value").is_finite()
        & (pl.col("selected_land_value") > 0)
    ).with_columns(
        (
            pl.col("land_value_2023") / pl.col("parcel_area_sqft") * 43_560
        ).alias("land_value_per_acre_2023"),
        (
            pl.col("selected_land_value") / pl.col("parcel_area_sqft") * 43_560
        ).alias("land_value_per_acre_2026"),
    ).with_columns(
        (
            pl.col("land_value_per_acre_2026")
            - pl.col("land_value_per_acre_2023")
        ).alias("land_value_per_acre_change")
    )
    values = comparison.to_pandas()
    result = parcels.merge(values, on="parcel_id", how="left")
    for column in (
        "land_value_per_acre_2023",
        "land_value_per_acre_2026",
        "land_value_per_acre_change",
    ):
        result.loc[~np.isfinite(result[column]), column] = np.nan
    return result


## 2. Resolve the prepared data

The aliases below name the inputs you will read. The data SDK resolves only the snapshots pinned by this project. Change prepared inputs through a reviewed data lock update, not by pointing at a moving latest file.


In [ ]:
from strongtowns_data import DataBuildSystem, DataLock, DataRepository
from strongtowns_detroit.repositories import data_repository
from strongtowns_graphics import GraphicBuildContext

requirements = (
        GraphicInput("parcels", "detroit.parcels.raw", "raw.geojson"),
        GraphicInput("assessments", "detroit.assessments.raw", "raw.geojson"),
        GraphicInput("history", "detroit.lvt-estimator-2023.raw", "raw.csv"),
        GraphicInput("boundary", "detroit.osm.basemap.raw", "detroit_boundary.geojson"),
        GraphicInput("water", "detroit.osm.basemap.raw", "detroit_water.geojson"),
        GraphicInput("roads", "detroit.base-units.streets.raw", "raw.geojson"),
    )
lock = DataLock.load(ROOT / "strongtowns-data.lock.json")
repository = DataRepository(DataBuildSystem.find(data_repository()))
paths, provenance = {}, {}
for requirement in requirements:
    try:
        reference = lock.asset(requirement.dataset_id)
        paths[requirement.alias] = repository.artifact(reference, requirement.artifact)
        if not paths[requirement.alias].is_file():
            raise FileNotFoundError(paths[requirement.alias])
        provenance[requirement.alias] = {
            "dataset": requirement.dataset_id,
            "artifact": requirement.artifact,
            "snapshot": reference.snapshot_id,
            "manifest_sha256": reference.manifest_sha256,
        }
    except (ValueError, KeyError, FileNotFoundError) as error:
        raise RuntimeError(
            f"Prepared input unavailable: {requirement.dataset_id}/{requirement.artifact}. "
            "Ask the data maintainer to restore the pinned snapshot in strongtowns-data; "
            "this notebook never fetches data or changes the lock."
        ) from error
context = GraphicBuildContext(paths)
provenance


## 3. Prepare and inspect the table

This follows the existing graphic’s data selection and calculations. The displayed rows are a preview; the graphic uses the full prepared table.


In [ ]:
frame = comparison_frame(
    context.input("parcels"),
    context.input("assessments"),
    context.input("history"),
)

comparable = int(frame["land_value_per_acre_2023"].notna().sum())

sources = (
    "Sources: City of Detroit 2023 residential LVT estimator, 2026 tentative "
    "assessment roll, and current parcel geometry.",
)

common_subtitle = (
    f"Assessor-recorded land value per acre · same {comparable:,} residential parcels"
)

basemap = load_detroit_basemap(
    context.input("boundary"), context.input("roads"), context.input("water")
)


In [ ]:
display(frame.drop(columns="geometry").head(8))


## 4. Build the graphic with the SDK

Edit `title`, `subtitle`, colors, legends, or explicit encodings here. Keep sources and descriptions accurate when changing data. The parcel and travel examples reuse existing project map-drawing helpers, then call SDK composition functions; the bar, point-map, and continuous-choropleth examples expose their renderer calls directly.


In [ ]:
map_2023 = parcel_choropleth_map(
    frame,
    value_column="land_value_per_acre_2023",
    scale=VALUE_SCALE,
    basemap=basemap,
    title=("Detroit residential land values", "in 2023"),
    subtitle=common_subtitle,
    legend_heading="LAND VALUE PER ACRE",
    sources=sources,
    description=(
        "City-recorded 2023 land value per acre for residential parcels "
        "that can also be matched to the 2026 tentative assessment roll."
    ),
    missing_label="Outside comparison cohort",
    missing_color=NO_DATA,
)

map_2026 = parcel_choropleth_map(
    frame,
    value_column="land_value_per_acre_2026",
    scale=VALUE_SCALE,
    basemap=basemap,
    title=("Detroit residential land values", "estimated for 2026"),
    subtitle=(
        f"Filtered estimate per acre · same {comparable:,} residential parcels"
    ),
    legend_heading="LAND VALUE PER ACRE",
    sources=sources,
    description=(
        "Estimated 2026 land value per acre after replacing only isolated "
        "fourfold spatial spikes without nearby similarly valued parcels."
    ),
    missing_label="Outside comparison cohort",
    missing_color=NO_DATA,
)

change = parcel_choropleth_map(
    frame,
    value_column="land_value_per_acre_change",
    scale=CHANGE_SCALE,
    basemap=basemap,
    title=("Change in Detroit residential", "land values, 2023–2026"),
    subtitle=(
        f"Change per acre · same {comparable:,} residential parcels"
    ),
    legend_heading="CHANGE IN LAND VALUE PER ACRE",
    sources=sources,
    description=(
        "Difference between the filtered 2026 estimate and City-recorded "
        "2023 land value per acre for the common residential parcel cohort."
    ),
    missing_label="Outside comparison cohort",
    missing_color=NO_DATA,
)

graphics = {
        "residential-land-value-2023": map_2023,
        "residential-land-value-2026": map_2026,
        "residential-land-value-change-2023-2026": change,
    }


## 5. Choose the Instagram format and preview

Feed uses 1080 × 1350; story uses 1080 × 1920 with the library’s established content padding. Changing this enum preserves the publishing policy.


In [ ]:
# Change to GraphicFormat.INSTAGRAM_STORY for a story-sized export.
TARGET = GraphicFormat.INSTAGRAM_POST
EXPORT_PNG = True  # SVG and HTML work without the rsvg-convert system tool.
target = GRAPHIC_FORMAT_SPECS[TARGET]


In [ ]:
# Preview exactly the composition used by the export below.
for name, graphic in graphics.items():
    print(name)
    svg = (render_graphic_canvas(
        graphic, canvas_aspect_ratio=target.aspect_ratio,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding,
    ) if target.content_aspect_ratio else render_graphic_svg(
        graphic, aspect_ratio=target.aspect_ratio,
    ))
    display(SVG(svg))
    print("Alt text:", graphic.description or graphic.title_text)


## 6. Export

PNG is ready for Instagram; SVG and HTML retain the composition for inspection. Copy the adjacent alt-text file when posting. Exports go to the ignored `projects/graphics/output/notebooks/` directory and can be regenerated. Inspect all pages before sharing.


In [ ]:
import json
import shutil

output_dir = GRAPHICS / "output/notebooks" / NOTEBOOK_NAME / TARGET.value
formats = ("html", "svg", "png") if EXPORT_PNG else ("html", "svg")
if EXPORT_PNG and shutil.which("rsvg-convert") is None:
    raise RuntimeError(
        "PNG export needs rsvg-convert (see README.md). "
        "Set EXPORT_PNG = False above and rerun the export to save SVG/HTML now."
    )
for name, graphic in graphics.items():
    files = write_graphic_bundle(
        output_dir, name, graphic,
        aspect_ratio=target.aspect_ratio, png_width=target.png_width,
        content_aspect_ratio=target.content_aspect_ratio,
        content_top_padding=target.content_top_padding, formats=formats,
    )
    (output_dir / f"{name}.alt.txt").write_text(
        graphic.description or graphic.title_text, encoding="utf-8"
    )
    for kind, path in files.items():
        print(f"{kind}: {path}")
# Keep the exact source identities alongside your exported graphics.
(output_dir / "sources.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8"
)


## Try the pattern on another question

Make a copy of this notebook. Start by changing editorial wording, then inspect the explicit input table before changing a field or grouping. Keep units, unknown records, source coverage, and denominators visible. Changing geographic scope or a legal threshold requires reviewing the method and claim, not just replacing the title.

Use the other notebooks to compare stacked bars, categorized points, continuous parcel maps, and mobile map compositions. Clear outputs before committing a notebook; put publishing changes back into the canonical definition.
